<div dir="rtl" align="right">

# بهینه‌سازها: مومنتوم، RMSProp و Adam

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## مشکل گرادیان کاهشی ساده
در دره‌های باریک (وقتی انحنا در جهت‌های مختلف خیلی فرق دارد)، گرادیان کاهشی ساده **زیگزاگ** می‌رود: در جهت تند نوسان می‌کند و در جهت کم‌شیب آهسته پیش می‌رود.

> [چهار بهینه‌ساز از یک نقطه شروع روی دره باریک $f=\frac12x^2+12.5y^2$. ستاره قرمز کمینه است.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_optimizers.png)

*چهار بهینه‌ساز از یک نقطه شروع روی دره باریک $f=\frac12x^2+12.5y^2$. ستاره قرمز کمینه است.*

## مومنتوم: توپی که سرعت می‌گیرد
به جای گام مستقیم، **سرعت** را به‌روز می‌کنیم که میانگین متحرک گرادیان‌های قبلی است:

$$v\leftarrow\beta v-\eta\nabla L\qquad\theta\leftarrow\theta+v$$

- در جهتی که گرادیان‌ها هم‌علامت‌اند، سرعت **جمع می‌شود**.
- در جهتی که علامت عوض می‌شود (زیگزاگ)، **خنثی می‌شوند**.

با $\beta=0.9$ سرعت نهایی تا $\dfrac{1}{1-\beta}=10$ برابر یک گام ساده می‌شود.

## مثال عددی مومنتوم
گرادیان ثابت $g=1$، $\eta=0.1$، $\beta=0.9$:

| گام | $v=0.9v-0.1$ | جابه‌جایی کل |
|---|---|---|
| 1 | −0.100 | −0.100 |
| 2 | −0.190 | −0.290 |
| 3 | −0.271 | −0.561 |
| 4 | −0.344 | −0.905 |
| ∞ | −1.000 | |

## RMSProp: نرخ یادگیری جدا برای هر پارامتر
میانگین متحرک **مربع** گرادیان را نگه می‌دارد و گام را بر جذر آن تقسیم می‌کند:

$$s\leftarrow\rho s+(1-\rho)(\nabla L)^2\qquad\theta\leftarrow\theta-\frac{\eta}{\sqrt{s}+\epsilon}\nabla L$$

پارامتری که گرادیان‌های بزرگ دارد گام کوچک‌تر می‌گیرد و برعکس؛ یعنی مقیاس‌بندی خودکار.

## Adam = مومنتوم + RMSProp
| گام | فرمول | معنی |
|---|---|---|
| 1 | $m\leftarrow\beta_1m+(1-\beta_1)g$ | میانگین گرادیان (جهت) |
| 2 | $s\leftarrow\beta_2s+(1-\beta_2)g^2$ | میانگین مربع گرادیان (مقیاس) |
| 3 | $\hat{m}=\dfrac{m}{1-\beta_1^t},\quad\hat{s}=\dfrac{s}{1-\beta_2^t}$ | تصحیح بایاس شروع از صفر |
| 4 | $\theta\leftarrow\theta-\eta\dfrac{\hat{m}}{\sqrt{\hat{s}}+\epsilon}$ | گام |

مقادیر پیش‌فرض: $\beta_1=0.9$، $\beta_2=0.999$، $\eta=0.001$، $\epsilon=10^{-8}$.

## مثال عددی: گام اول Adam
با $g=4$ و $\eta=0.1$:

| کمیت | محاسبه | مقدار |
|---|---|---|
| $m$ | $0.1\times4$ | 0.4 |
| $s$ | $0.001\times16$ | 0.016 |
| $\hat{m}$ | $0.4/(1-0.9)$ | 4 |
| $\hat{s}$ | $0.016/(1-0.999)$ | 16 |
| گام | $0.1\times4/\sqrt{16}$ | 0.1 |

> نتیجه جالب: گام اول Adam همیشه تقریباً برابر $\eta$ است، **مستقل از بزرگی گرادیان**. اگر گرادیان 4000 بود هم گام 0.1 می‌شد. به همین دلیل Adam به مقیاس تابع خطا حساس نیست و نرخ یادگیری پیش‌فرض آن اغلب خوب کار می‌کند.

## مقایسه
| بهینه‌ساز | حافظه اضافه | نقطه قوت | نقطه ضعف |
|---|---|---|---|
| SGD | هیچ | ساده؛ تعمیم خوب | کند در دره‌ها؛ حساس به η |
| SGD + مومنتوم | v | سریع‌تر؛ انتخاب رایج در بینایی ماشین | یک ابرپارامتر بیشتر |
| RMSProp | s | مناسب شبکه‌های بازگشتی | بدون مومنتوم |
| Adam | m و s | پیش‌فرض خوب؛ تنظیم کم | گاهی تعمیم کمی ضعیف‌تر |
| AdamW | m و s | Adam با weight decay درست؛ استاندارد ترنسفورمرها | |

## برنامه نرخ یادگیری
| برنامه | رفتار | کاربرد |
|---|---|---|
| ثابت | همیشه η | ساده |
| پله‌ای | هر چند epoch تقسیم بر 10 | شبکه‌های کانولوشنی کلاسیک |
| کسینوسی | کاهش نرم تا صفر | آموزش‌های مدرن |
| Warmup | شروع از صفر و افزایش تدریجی | ترنسفورمرها |
| کاهش روی فلات | وقتی خطای اعتبارسنجی بهتر نمی‌شود | عمومی |



</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

f    = lambda p: 0.5 * p[0] ** 2 + 12.5 * p[1] ** 2       # a narrow valley: 25 times steeper along y
grad = lambda p: np.array([p[0], 25 * p[1]])

def run(name, steps=80):
    p = np.array([-3.5, 1.5]); path = [p.copy()]
    v = np.zeros(2); s = np.zeros(2); m = np.zeros(2)
    for t in range(1, steps + 1):
        g = grad(p)
        if name == "GD":
            p = p - 0.07 * g                      # must stay below 2/25 = 0.08 or it diverges
        elif name == "Momentum":
            v = 0.8 * v - 0.02 * g; p = p + v
        elif name == "RMSProp":
            s = 0.9 * s + 0.1 * g ** 2; p = p - 0.1 * g / (np.sqrt(s) + 1e-8)
        elif name == "Adam":
            m = 0.9 * m + 0.1 * g; s = 0.999 * s + 0.001 * g ** 2
            p = p - 0.2 * (m / (1 - 0.9 ** t)) / (np.sqrt(s / (1 - 0.999 ** t)) + 1e-8)
        path.append(p.copy())
    return np.array(path)

u, w = np.meshgrid(np.linspace(-4, 1.5, 200), np.linspace(-2, 2, 200))
plt.contour(u, w, 0.5 * u ** 2 + 12.5 * w ** 2, levels=12, colors="lightgray")
for name in ["GD", "Momentum", "RMSProp", "Adam"]:
    path = run(name)
    plt.plot(path[:, 0], path[:, 1], marker="o", ms=2, label=name)
    values = np.array([f(q) for q in path])
    hit = np.argmax(values < 0.01) if (values < 0.01).any() else "never"
    print(f"{name:9s} f after 80 steps = {values[-1]:.2e}   first step with f < 0.01: {hit}")
plt.legend(); plt.title("Optimizers on a narrow valley"); plt.show()

# Momentum with a constant gradient: the velocity approaches -eta / (1 - beta)
v = 0.0
for step in range(1, 5):
    v = 0.9 * v - 0.1 * 1.0
    print(f"step {step}: v = {v:.3f}")

# The first Adam step has size eta, whatever the gradient
for g in [0.004, 4.0, 4000.0]:
    m = 0.1 * g; s = 0.001 * g ** 2
    print(f"g = {g:8}: first Adam step = {0.1 * (m / 0.1) / np.sqrt(s / 0.001):.4f}")

<div dir="rtl" align="right">

### تمرین

با $\beta=0.5$، $\eta=0.1$ و گرادیان ثابت $g=2$، سرعت مومنتوم را برای سه گام حساب کن. سرعت نهایی چقدر است؟

</div>

In [ ]:
# Your answer here

<div dir="rtl" align="right">

### تمرین

در آزمایشگاه گرادیان کاهشی (درس قبل) دره باریک را با $\eta=0.15$ بدون مومنتوم و سپس با $\beta=0.8$ اجرا کن.

</div>

In [ ]:
# Your answer here